# 使用 Promptfoo 的模型分级评估

**注意：本课程位于包含相关代码文件的文件夹中。如果您想跟随并自己运行评估，请下载整个文件夹**


到目前为止，我们只编写了代码分级评估。在可能的情况下，代码分级评估是最简单、成本最低的评估方式。它们提供基于预定义标准的明确、客观的评估，非常适合可以简化为精确匹配、数值比较或其他可编程逻辑的任务。问题在于，代码分级评估只能对某些类型的输出进行评分，主要是可以简化为精确匹配、数字比较或可编程逻辑的内容。

然而，许多现实世界的语言模型应用需要更细致的评估。假设我们想构建一个用于中学教室的聊天机器人。我们可能希望评估输出，确保它们使用适合年龄的语言，保持教育基调，避免回答非学术问题，或以适合中学生的复杂度提供解释。这些标准是主观的且依赖于上下文，使得用传统代码方法进行评估变得困难。这时，模型分级评估可以提供帮助！

模型分级评估利用大型语言模型的能力，根据更复杂、更细致的标准评估输出。通过使用另一个模型作为评估者，我们可以利用与生成原始响应相同的语言理解和上下文感知能力。这种方法使我们能够创建更复杂的评估指标，考虑语气、相关性、适当性甚至创造力等因素——这些通常是代码分级系统无法触及的方面。

模型分级评估的核心思想是将评估本身视为一项自然语言处理任务。我们向评估模型提供以下内容的某种组合：

* 原始提示或问题
* 我们想要评估的模型生成的响应
* 一套评估标准或指南
* 如何评估和评分响应的说明

这种方法允许对输出进行更全面的评估，不仅考虑事实准确性，还考虑风格元素、对特定指南的遵守情况以及响应在其预期使用上下文中的整体质量。

常见的模型分级评估技术包括要求模型：

* 这个响应的道歉程度如何？
* 给定提供的上下文，响应在事实层面是否准确？
* 这个响应是否过多提及其上下文/信息？
* 这个响应是否真正适当地回答了问题？
* 这个输出对我们的语气/品牌/风格指南的遵守程度如何？

在本课程中，我们将使用 promptfoo 编写我们自己的简单模型分级评估。

---

## 使用 Promptfoo 进行模型分级评估

与 promptfoo 中的大多数事情一样，编写模型分级评估有多种有效方法。在本课程中，我们将看到最简单的方式：利用内置断言。在下一课中，我们将看到如何编写我们自己的自定义模型分级断言函数。

首先，我们将使用一个名为 `llm-rubric` 的内置断言，这是 promptfoo 用于"LLM 作为评判者"评估的通用评分器。使用它非常简单，只需将其添加到您的 `promptfooconfig.yaml` 文件中：

```yaml
assert:
  - type: llm-rubric
    # 我们想要用作评分员的模型
    provider: anthropic:messages:claude-3-opus-20240229
    # 指定评分标准：
    value: Is not apologetic
```
上述断言将使用 Claude 3 Opus 根据响应是否具有道歉性进行评分。

让我们在自己的评估中尝试使用 `llm-rubric`！

---

## 编写我们自己的评估
在本课程中，我们将专注于评估面向初中生学术助手的提示词。我们正在构建一个聊天机器人，它应该回答与学校科目相关的问题，但应该避免回答无关的问题。我们将从如下简单提示词开始：

```
You are an interactive tutor assistant for middle school children.
Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
Only answer questions related to middle school academics.
This is the student question: {{question}}
```

我们将编写一个模型分级评估，以确保模型拒绝回答诸如"我怎样才能让父母给我买手机？"或"有哪些有趣的免费视频游戏？"之类的问题。从理论上讲，我们可以尝试编写一个使用正则表达式查找关键拒绝短语（如"对不起"、"我无法帮助"等）的代码分级评估，但这是一种非常脆弱的方法。使用模型来评估响应可以获得更好的结果。

--- 

## 初始化 Promptfoo

一如既往，第一步是初始化 promptfoo。在一个新文件夹中，运行以下命令：

```bash
npx promptfoo@latest init
```

---

## 基本配置

为了使事情更简单，我们不会将提示词放在单独的文件中，就像之前所做的那样。我们将直接把提示词和测试写在 `promptfooconfig.yaml` 文件中。删除文件的现有内容并替换为：

```yaml
description: "School Assistant Eval"

prompts:
  - >- 
    You are an interactive tutor assistant for middle school children.
    Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
    Only answer questions related to middle school academics.
    This is the student question {{question}}

providers: [anthropic:messages:claude-3-5-sonnet-20240620]
  # - anthropic:messages:claude-3-haiku-20240307

defaultTest:
  assert:
    - type: llm-rubric
      provider: anthropic:messages:claude-3-opus-20240229
      value: 'Refuses to answer the question and instead redirects to academic topics'

tests:
  - vars:
      question: What's the best free mobile video game?
  - vars:
      question: How can I convince my parents to let me stay up later?
  - vars:
      question: What are some fun theme ideas for a sleepover?
  - vars:
      question: What are some good prank ideas that won't get me in trouble?
  - vars:
      question: Who are some famous left footed soccer players?
```

上述配置包含以下内容：

* `prompts` - 包含我们想要评估的初始基本提示词。请记住，我们经常将提示词放在 Python 文件中，但我们也可以选择将它们直接内联在配置文件中。
* `providers` - 我们告诉 promptfoo 我们想要通过 Claude 3.5 Sonnet 运行我们的提示词
* `defaultTest` - 在 promptfoo 运行的每个测试中，我们想要使用 `llm-rubric` 对其进行评分，以确保输出拒绝回答问题并将其重定向到学术主题。在这种情况下，我们告诉 promptfoo 使用 Claude 3 Opus 执行评分。
* `tests` - 我们想要运行的一组测试。我们经常在 CSV 文件中指定这些，但也可以在 YAML 文件中内联编写测试。所有这些测试都包含我们希望模型拒绝回答的问题。

下一步是使用以下命令运行我们的评估：

```bash
npx promptfoo@latest eval
```

这是我们首次运行此评估时生成的输出：



我们的提示词与我们大多数评估数据集配合良好（尽管这是一个非常小的数据集），但看起来模型乐意回答有关足球运动员的问题。以下来自 promptfoo Web 界面的截图展示了模型的响应以及评分模型的评分逻辑：



让我们尝试添加第二个更详细的提示词，看看是否能促使模型坚持学术主题。

---

## 第二个提示词
让我们添加第二个提示词，其中包含关于模型应该讨论哪些主题的更精确指南：

>  You are an interactive tutor assistant for middle school children.
    Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
    Only answer questions related to middle school academics.
    Acceptable topics you should respond to include: math, reading, science, foreign languages, social students, and art.
    Refuse to answer questions not related to the above topics in an academic context.
    This is the student question: {{question}}

更新 `promptfooconfig.yaml` 文件，如下所示：

```yaml
description: "School Assistant Eval"

prompts:
  - >- 
    You are an interactive tutor assistant for middle school children.
    Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
    Only answer questions related to middle school academics.
    This is the student question: {{question}}
  - >- 
    You are an interactive tutor assistant for middle school children.
    Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
    Only answer questions related to middle school academics.
    Acceptable topics you should respond to include: math, reading, science, foreign languages, social students, and art.
    Refuse to answer questions not related to the above topics in an academic context.
    This is the student question: {{question}}


providers: [anthropic:messages:claude-3-5-sonnet-20240620]

defaultTest:
  assert:
    - type: llm-rubric
      provider: anthropic:messages:claude-3-opus-20240229
      value: 'Refuses to answer the question and instead redirects to academic topics'

tests:
  - vars:
      question: What's the best free mobile video game?
  - vars:
      question: How can I convince my parents to let me stay up later?
  - vars:
      question: What are some fun theme ideas for a sleepover?
  - vars:
      question: What are some good prank ideas that won't get me in trouble?
  - vars:
      question: Who are some famous left footed soccer players?
```

我们现在有两个正在评估的提示词！让我们使用以下命令再次运行评估：

```bash
npx promptfoo@latest eval
```



这是我们得到的结果：



看起来我们对第二个提示词的修改产生了效果！请注意，这个评估数据集太小了，在现实世界中毫无用处。

---

## 道歉评估

在仔细观察模型输出时，我们注意到大多数输出都以"对不起"或"我道歉"等道歉语开头。这对用户来说不是理想的体验，所以我们决定尝试改进这一点！我们希望评估第三个提示词：

> You are an interactive tutor assistant for middle school children.
    Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
    Only answer questions related to middle school academics.
    Acceptable topics you should respond to include: math, reading, science, foreign languages, social students, and art.
    Refuse to answer questions not related to the above topics in an academic context.
    Do not apologize or use an apologetic tone when refusing.  Instead, gently nudge students to focus on school-related topics.
    This is the student question: {{question}}

上述提示词明确告诉模型避免道歉，而是专注于温和地引导学生专注于学术主题。

接下来，让我们添加第二个 `llm-rubric` 断言来测试模型输出是否具有道歉性。将 `promptfooconfig.yaml` 更新为如下内容：

```yaml
description: "School Assistant Eval"

prompts:
  - >- 
    You are an interactive tutor assistant for middle school children.
    Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
    Only answer questions related to middle school academics.
    This is the student question: {{question}}
  - >- 
    You are an interactive tutor assistant for middle school children.
    Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
    Only answer questions related to middle school academics.
    Acceptable topics you should respond to include: math, reading, science, foreign languages, social students, and art.
    Refuse to answer questions not related to the above topics in an academic context.
    This is the student question: {{question}}
  - >- 
    You are an interactive tutor assistant for middle school children.
    Students will ask you a question and your job is to respond with explanations that are understandable to a middle school audience.
    Only answer questions related to middle school academics.
    Acceptable topics you should respond to include: math, reading, science, foreign languages, social students, and art.
    Refuse to answer questions not related to the above topics in an academic context.
    Do not apologize or use an apologetic tone when refusing.  Instead, gently nudge students to focus on school-related topics.
    This is the student question: {{question}}


providers: [anthropic:messages:claude-3-5-sonnet-20240620]

defaultTest:
  assert:
    - type: llm-rubric
      provider: anthropic:messages:claude-3-opus-20240229
      value: 'Refuses to answer the question and instead redirects to academic topics'
    - type: llm-rubric
      provider: anthropic:messages:claude-3-opus-20240229
      value: 'Is not apologetic'

tests:
  - vars:
      question: What's the best free mobile video game?
  - vars:
      question: How can I convince my parents to let me stay up later?
  - vars:
      question: What are some fun theme ideas for a sleepover?
  - vars:
      question: What are some good prank ideas that won't get me in trouble?
  - vars:
      question: Who are some famous left footed soccer players?
```

我们现在有三个正在测试的提示词。对于每个测试用例，我们使用模型对两个单独方面进行评分：
* 模型应该拒绝回答问题
* 模型不应该道歉

让我们运行评估：

```bash
npx promptfoo@latest eval
```


这是我们得到的结果：



正如预期的那样，前两个提示词在道歉断言上失败了，但第三个提示词似乎有效！

让我们使用以下命令启动 Web 界面：

```bash
npx promptfoo@latest view
```



请记住，我们可以点击放大镜图标查看每个模型输出和相应断言评分的更多详细信息。让我们仔细看看第一行中的第二个条目：



我们可以看到输出通过了原始模型分级断言，并且确实拒绝回答无关问题。我们还可以看到输出未能通过我们添加的第二个断言，因为"响应以'对不起'开头，这是一个道歉短语。"

现在让我们放大第一行中的第三个条目：



此输出通过了两项断言！

**请记住，这个数据集对于实际评估来说太小了。**

Promptfoo 内置的模型分级断言非常有用，但在某些情况下，我们可能需要更多地控制精确的模型分级指标和流程。在下一课中，我们将了解如何定义我们自己的自定义模型评分函数！